# Diffie-Hellman

Some recalls on the Discrete logarithm Problem (DLP).  
We choose a prime number $p$ and a *primitive root* $g$ modulo $p$ i.e. a generator of the multiplicative group $\mathbb{Z}_p^* = \{g^0, g^1, g^2, \ldots, g^{p-1} \}$.

- $p$: prime number of $n$ bits

- $g$: primitive root

- $q$: order of $g$, this means $q = |\mathbb{Z}_p^*| = p - 1$

**In such setting**:

$$
g^x \quad \text{easy to compute}
$$

$$
x \leftarrow g^x \quad \text{hard to recover}
$$


In [2]:
# Starting with this prime
p = 23
g_candidates = []
# Find primitive roots for p
for i in range(1, p):
    generated_elements = []
    for exp in range(1,p):
        value = pow(i,exp,p)
        if not value in generated_elements:
            generated_elements.append(value)
    if(len(generated_elements) == p - 1):
        g_candidates.append(i)

print(f"Primitive roots of p: {g_candidates}")

Primitive roots of p: [5, 7, 10, 11, 14, 15, 17, 19, 20, 21]


In [3]:
# Choose a g 
g = 5

# Pick an exponent and compute g^x mod p
x = 19
u = pow(g,x,p)
print(f"u = {u}")

u = 7


In [4]:
# Starting from u and g find x
for i in range(1,p):
    value = pow(g,i,p)
    if(value == u):
        print(f"DLP(u) = {i}")


DLP(u) = 19


<details>
<summary>💡 Show solution</summary>

- Primitive roots of p $\[5,7,10,11,14,15,17,19,20,21]\$

</details>

## Key Exchange

We can define the Diffie-Hellman Key Exchange:

1. We assume that the description of $g$ and $q$, is a system parameter that is generated once and for all at system setup time and shared by all parties involved.
2. Alice computes $\alpha \xleftarrow{R} \mathbb{Z}_p$, $u \leftarrow g^\alpha$ and send $u$ to Bob.
3. Bob computes $\beta \xleftarrow{R} \mathbb{Z}_p$, $v \leftarrow g^\beta$ and send $v$ to Alice.

4. Both parties computes respectivelly $v^\alpha$ anb $u^\beta$, so the shared secret is $w=v^\alpha = g^{\alpha\beta} = u^\beta$.


<div style="display:flex; text-align:center; justify-content: center; gap: 30px;">
  <div style="width: 400px; text-align: center;">
    <img src="imgs/DH-KE.png" style="width: 200; height: auto;">
  </div>

</div>



In [5]:
TOY_PARAMS = {

    "TOY-10": {
        "bits": 10,
        "p": 1009,
        "g": 11,
    },

    "TOY-14": {
        "bits": 14,
        "p": 12289,
        "g": 11,
    },
    
    "TOY-16": {
        "bits": 17,
        "p": 65537,
        "g": 3,
    },

    "TOY-24": {
        "bits": 24,
        "p": 16777259,
        "g": 2,
    },

    "TOY-32": {
        "bits": 32,
        "p": 4294967291,
        "g": 2,
    },
}
##############################################################################
# Select public informations
##############################################################################
GROUP = "TOY-24"

if GROUP in TOY_PARAMS:
    gruppo = TOY_PARAMS[GROUP]
    p = gruppo["p"]
    g = gruppo["g"]
    dimensione_bit = gruppo["bits"]
    
    #print(f"[Profilo Caricato]: {GROUP} ({dimensione_bit} bit)")
    print(f"[Public Values] (p = {p}, g = {g})\n")
else:
    print(f"Errore: Il profilo '{GROUP}' non è presente nella tabella.")


##############################################################################
# Key Exchange
##############################################################################
alpha = p // 2
u = pow(g, alpha, p)

print(f"[ALICE] Secret Key, alpha = {alpha}")
print(f"[ALICE] Public Value, g^x = {u} ")
print()
print(f"[ALICE] ---- u ----> BOB")
print()

beta = 15
v = pow(g, beta, p)
print(f"[BOB] Secret Key, beta = {beta}")
print(f"[BOB] Public Value, g^y = {v} ")
print()
print(f"[ALICE] <--- v ----- BOB")
print()

# Both compute the shared secret
K_A = pow(v, alpha, p)
K_B = pow(u, beta, p)
print(f"[ALICE] Shared secret, (g^y)^x = {K_A} ")
print(f"[ALICE] Shared secret, (g^x)^y = {K_B} ")


[Public Values] (p = 16777259, g = 2)

[ALICE] Secret Key, alpha = 8388629
[ALICE] Public Value, g^x = 16777258 

[ALICE] ---- u ----> BOB

[BOB] Secret Key, beta = 15
[BOB] Public Value, g^y = 32768 

[ALICE] <--- v ----- BOB

[ALICE] Shared secret, (g^y)^x = 16777258 
[ALICE] Shared secret, (g^x)^y = 16777258 


## Attacks

Ora proviamo a pensare come un attaccante e vediamo come sono i parametri scelti.

- Attacco brute force (per far vedere come scegliere l'esponente, all'incirca a meta') richiamando quello che abbiamo visto nella lezione prima
- MITM (problema strutturale dovuta al fatto che e' un key exchange anonimo)

### Brute Force

E quindi qui facciamo vedere quali sono i parametri veri 

In [6]:
####################################################################
# Try to brute force DLP(u)
####################################################################
import time

def discrete_log_bruteforce(g, A, p):
    for x in range(p):
        if pow(g, x, p) == A:
            return x
    return None

start = time.perf_counter()
recovered = discrete_log_bruteforce(g, u, p)
elapsed = time.perf_counter() - start

print(f"Recovered: {recovered}")
print(f"Time: {elapsed:.2f} seconds")

# Find the shared secret
recovered_ss = pow(v,recovered,p)
print(f"Eve recovered the following secret {recovered_ss}")


Recovered: 8388629
Time: 5.06 seconds
Eve recovered the following secret 16777258


---

To avoid brute-forcing we have to choose parameters that make the problem very hard.
Examples of real parameters of DH Groups used in network protocols are like this: 

- [RFC3526](https://www.rfc-editor.org/rfc/rfc3526.txt)

In [7]:
from utils import *

# Available groups: MODP-1536 | MODP-2048 | MODP-3072 | MODP-4096 | MODP-6144 | MODP-8192
GROUP = "MODP-2048"

print("=" * 70)
describe_group(GROUP)
print("=" * 70)

if GROUP in DH_PARAMS:
    gruppo = DH_PARAMS[GROUP]
    p = gruppo["p"]
    g = gruppo["g"]
    dimensione_bit = gruppo["bits"]
    
    print(f"[Public Values]\np = {p}\ng = {g}")
else:
    print(f"Errore: Il profilo '{GROUP}' non è presente nella tabella.")

[Selected group]: MODP-2048
	RFC group: 14
	Bits: 2048
	Decimal digits: 617
	Generator: 2
[Public Values]
p = 32317006071311007300338913926423828248817941241140239112842009751400741706634354222619689417363569347117901737909704191754605873209195028853758986185622153212175412514901774520270235796078236248884246189477587641105928646099411723245426622522193230540919037680524235519125679715870117001058055877651038861847280257976054903569732561526167081339361799541336476559160368317896729073178384589680639671900977202194168647225871031411336429319536193471636533209717077448227988588565369208645296636077250268955505928362751121174096972998068410554359584866583291642136218231078990999448652468262416972035911852507045361090559
g = 2


### MITM

Codice per fare attacchi MITM


# ElGamal


## Key Generation

- Si sceglie una chiave privata $x \xleftarrow{R} \mathbb{Z}_{p-1}$
- Si calcola la chiave pubblica $y \leftarrow g^x \mod p$

$$pk = \{p, g, y\}, \qquad sk = \{x\}$$


## Encryption

Per cifrare un messaggio $m \in \mathbb{Z}_p^*$ con la chiave pubblica $pk = \{p, g, y\}$:

- Si sceglie un **nonce casuale** $k \xleftarrow{R} \mathbb{Z}_{p-1}$ (diverso ad ogni cifratura!)
- Si calcola $\;c_1 \leftarrow g^k \mod p$
- Si calcola $\;c_2 \leftarrow m \cdot y^k \mod p$

Il ciphertext è la coppia $\;c = (c_1, c_2)$


## Decryption

Per decifrare $c = (c_1, c_2)$ con la chiave privata $sk = \{x\}$:

- Si calcola il segreto condiviso $\;s \leftarrow c_1^x \mod p$ (Bob può calcolarlo perché conosce $x$, senza bisogno di $k$)
- Si recupera il messaggio $\;m \leftarrow c_2 \cdot s^{-1} \mod p$

L'inverso moltiplicativo $s^{-1} \mod p$ si calcola con il **piccolo teorema di Fermat**, dato che $p$ è primo:
$$s^{-1} \equiv s^{p-2} \pmod p$$


In [12]:
# Implement Keygen | Encrypt | Decrypt

GROUP = "TOY-24"

if GROUP in TOY_PARAMS:
    gruppo = TOY_PARAMS[GROUP]
    p = gruppo["p"]
    g = gruppo["g"]
    dimensione_bit = gruppo["bits"]
    print(f"[Public Values] (p = {p}, g = {g})\n")
else:
    print(f"Errore: Il profilo '{GROUP}' non è presente nella tabella.")

##############################################################################
# ElGamal Keygen
##############################################################################
def elgamal_keygen(p, g):
    x = random.randint(1, p - 2)   # chiave privata
    y = pow(g, x, p)               # chiave pubblica
    return x, y

def elgamal_encrypt(m, p, g, y):
    k = random.randint(1, p - 2)  
    c1 = pow(g, k, p)
    s = pow(y, k, p)                
    c2 = (m * s) % p
    return c1, c2
    
def elgamal_decrypt(c1, c2, p, x):
    s = pow(c1, x, p)               
    s_inv = pow(s, p - 2, p)       
    m = (c2 * s_inv) % p
    return m

x, y = elgamal_keygen(p, g)
msg = 42  # il messaggio deve essere un intero in Z_p^*, m < p

print(f"[BOB] Secret Key, x = {x}")
print(f"[BOB] Public Value, y = g^x mod p = {y}")

c1, c2 = elgamal_encrypt(msg, p, g, y)

print(f"[ALICE] Messaggio in chiaro, m = {msg}")
print(f"[ALICE] Ciphertext, (c1, c2) = ({c1}, {c2})")
print()
print(f"[ALICE] ---- (c1, c2) ----> BOB")


recovered = elgamal_decrypt(c1, c2, p, x)

print(f"[BOB] Shared secret, s = c1^x mod p = {pow(c1, x, p)}")
print(f"[BOB] Recovered message, m = {recovered}")

assert recovered == msg

[Public Values] (p = 16777259, g = 2)

[BOB] Secret Key, x = 121419
[BOB] Public Value, y = g^x mod p = 16408482
[ALICE] Messaggio in chiaro, m = 42
[ALICE] Ciphertext, (c1, c2) = (1544831, 8971276)

[ALICE] ---- (c1, c2) ----> BOB
[BOB] Shared secret, s = c1^x mod p = 1811436
[BOB] Recovered message, m = 42
